# Pipeline de Características para Predicción de Suscripción Bancaria

## Introducción

Este notebook tiene como propósito cargar los datos crudos del dataset de marketing bancario, preprocesarlos, realizar una ingeniería de características básica (escalado y codificación one-hot) y finalmente guardar las características procesadas. Este proceso es un paso fundamental antes del entrenamiento de cualquier modelo de machine learning, asegurando que los datos estén en un formato adecuado y optimizado.

## 1. Importar Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import os
import zipfile

## 2. Cargar Datos

Se cargarán los datos desde el archivo `bank-full.csv`. Si el archivo no existe localmente en una carpeta `./data/`, se intentará descargar y descomprimir.

In [ ]:
# Definir el directorio de datos y la ruta del archivo
data_dir = './data/'
csv_file_path = os.path.join(data_dir, 'bank-full.csv')
zip_file_path = os.path.join(data_dir, 'bank_marketing.zip')

# Crear el directorio de datos si no existe
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f"Directorio '{data_dir}' creado.")

# Verificar si el archivo CSV existe, si no, descargar y descomprimir
if not os.path.exists(csv_file_path):
    print(f"'{csv_file_path}' no encontrado. Intentando descargar y descomprimir...")
    # Comandos para descargar y descomprimir (pueden necesitar ! si se ejecutan en un entorno interactivo)
    # En un script de Python puro, se usaría requests y zipfile directamente.
    import subprocess
    try:
        subprocess.run(["wget", "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip", "-O", zip_file_path], check=True)
        print("Archivo ZIP descargado exitosamente.")
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # Extraer solo bank-full.csv al directorio de datos
            zip_ref.extract('bank-full.csv', path=data_dir)
        print(f"Archivo 'bank-full.csv' extraído a '{data_dir}'.")
        # Opcional: eliminar el archivo zip después de la extracción
        # os.remove(zip_file_path)
    except subprocess.CalledProcessError as e:
        print(f"Error durante la descarga: {e}")
    except FileNotFoundError: # Por si wget no está instalado
        print("Error: El comando 'wget' no fue encontrado. Por favor, descarga y descomprime el archivo manualmente en la carpeta './data/'.")
    except Exception as e:
        print(f"Ocurrió un error inesperado durante la descarga o extracción: {e}")
else:
    print(f"Archivo '{csv_file_path}' encontrado.")

# Cargar el dataset
try:
    df = pd.read_csv(csv_file_path, sep=';')
    print("\nDataset cargado exitosamente:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: El archivo '{csv_file_path}' aún no se encuentra. Verifica la descarga o la ruta.")
    df = None # Asegura que df es None si la carga falla

## 3. Preprocesamiento

### 3.1 Variable Objetivo
Convertir la variable objetivo `y` a formato binario: `1` para 'yes' y `0` para 'no'.

In [ ]:
if df is not None:
    df['y'] = df['y'].map({'yes': 1, 'no': 0})
    print("Variable objetivo 'y' convertida a binario:")
    display(df[['y']].head())

### 3.2 Definir Tipos de Columnas
Identificar columnas numéricas para escalado y categóricas para codificación one-hot.

In [ ]:
if df is not None:
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    # Remover la variable objetivo de las columnas numéricas a transformar
    if 'y' in numerical_cols:
        numerical_cols.remove('y')
    
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    
    print(f"Columnas numéricas para escalar: {numerical_cols}")
    print(f"Columnas categóricas para One-Hot Encoding: {categorical_cols}")

### 3.3 Manejo de Valores 'unknown'
Las columnas categóricas pueden contener valores 'unknown'. `OneHotEncoder` tratará estos valores como una categoría más si están presentes durante el `fit`. Si se encuentran nuevos valores 'unknown' (o cualquier otra categoría nueva) durante el `transform` (en datos no vistos), `handle_unknown='ignore'` hará que estas nuevas categorías se codifiquen como un vector de ceros para todas las nuevas columnas generadas por esa característica, evitando errores. Esto es útil si el conjunto de datos de producción podría tener categorías no vistas en el entrenamiento.

### 3.4 Crear Transformadores y Aplicar `ColumnTransformer`

In [ ]:
if df is not None and numerical_cols is not None and categorical_cols is not None:
    # Crear transformadores
    numerical_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False) # sparse_output=False para obtener un array denso
    
    # Crear el ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols)
        ],
        remainder='passthrough' # Mantener columnas no especificadas (en este caso, ninguna además de 'y' que ya separamos)
    )
    
    # Separar características (X) y objetivo (y_target) antes de aplicar el preprocesador
    X = df.drop('y', axis=1)
    y_target = df['y']
    
    # Aplicar el preprocesador a las características X
    X_processed_array = preprocessor.fit_transform(X)
    
    # Obtener los nombres de las nuevas columnas después del OneHotEncoding
    # El método get_feature_names_out está disponible en scikit-learn >= 0.24
    try:
        # Para scikit-learn >= 1.0 (aproximadamente)
        cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
    except AttributeError:
        # Para versiones más antiguas, puede ser necesario acceder a categories_ y construir los nombres manualmente
        # Esto es un fallback, la sintaxis exacta puede variar ligeramente entre versiones más antiguas
        print("Usando fallback para get_feature_names_out. Considera actualizar scikit-learn.")
        cat_feature_names = []
        for i, col in enumerate(categorical_cols):
            for cat_val in preprocessor.named_transformers_['cat'].categories_[i]:
                cat_feature_names.append(f"{col}_{cat_val}")
                
    # Combinar nombres de columnas numéricas y categóricas transformadas
    processed_col_names = numerical_cols + list(cat_feature_names)
    
    # Convertir el array procesado de vuelta a un DataFrame
    X_processed_df = pd.DataFrame(X_processed_array, columns=processed_col_names, index=X.index)
    
    print("\nCaracterísticas procesadas (primeras 5 filas):")
    display(X_processed_df.head())
    print(f"\nDimensiones del DataFrame de características procesadas: {X_processed_df.shape}")

## 4. División de Datos (Opcional)

Para un pipeline de características, típicamente se procesa todo el conjunto de datos disponible y se guarda. La división en conjuntos de entrenamiento y prueba se realiza comúnmente en el pipeline de entrenamiento del modelo. Aquí, ya hemos separado `X_processed_df` (características) y `y_target` (variable objetivo).

## 5. Almacenamiento de Características (Simulación de Feature Store)

Concatenamos las características procesadas y la variable objetivo, y luego las guardamos en un archivo Parquet. Este formato es eficiente para el almacenamiento y la lectura.

In [ ]:
if 'X_processed_df' in locals() and 'y_target' in locals():
    # Concatenar características procesadas y la variable objetivo
    final_processed_df = pd.concat([X_processed_df, y_target], axis=1)
    
    # Definir la ruta para el archivo Parquet
    processed_file_path = os.path.join(data_dir, 'processed_bank_data.parquet')
    
    # Guardar el DataFrame en formato Parquet
    try:
        final_processed_df.to_parquet(processed_file_path, index=False)
        print(f"\nDatos procesados guardados exitosamente en: '{processed_file_path}'")
        print("\nPrimeras filas del DataFrame guardado:")
        display(final_processed_df.head())
    except Exception as e:
        print(f"Error al guardar el archivo Parquet: {e}")
else:
    print("\nNo se pudieron guardar los datos procesados porque 'X_processed_df' o 'y_target' no están definidos.")
    print("Esto puede ocurrir si la carga inicial del dataset falló.")

### Nota sobre Feature Stores:

En un entorno de producción, estas características se guardarían en un **Feature Store** gestionado, como [Hopsworks](https://www.hopsworks.ai/), [AWS SageMaker Feature Store](https://aws.amazon.com/sagemaker/feature-store/), [Google Vertex AI Feature Store](https://cloud.google.com/vertex-ai/docs/featurestore) o [Feast](https://feast.dev/).

Un Feature Store ofrece ventajas significativas:
*   **Centralización:** Un lugar único para almacenar y acceder a características curadas.
*   **Versionado:** Permite rastrear cambios en las características y revertir a versiones anteriores.
*   **Compartición y Reutilización:** Facilita que diferentes equipos y modelos utilicen las mismas características consistentes.
*   **Monitorización:** Ayuda a detectar problemas como el *feature drift* (cambio en la distribución de las características a lo largo del tiempo).
*   **Consistencia Online/Offline:** Asegura que las mismas transformaciones de características se apliquen tanto para el entrenamiento de modelos (offline) como para la inferencia en tiempo real (online).

## 6. Automatización (Comentario)

Este notebook está diseñado para ser ejecutado automáticamente (por ejemplo, cada hora, diariamente o según la frecuencia de actualización de los datos fuente) usando herramientas de orquestación de flujos de trabajo como [Apache Airflow](https://airflow.apache.org/), [Kubeflow Pipelines](https://www.kubeflow.org/docs/components/pipelines/), o incluso GitHub Actions para proyectos más simples.

La acción de GitHub (o cualquier otro sistema de automatización) se encargaría de:
1.  Obtener la versión más reciente del código del pipeline (este notebook o su versión en script .py).
2.  Ejecutar el pipeline para procesar los datos crudos más recientes.
3.  Almacenar las características actualizadas en el Feature Store o en la ubicación de almacenamiento designada.
4.  Registrar la ejecución, manejar errores y notificar según sea necesario.

## 7. Generación de `requirements.txt`

Ejecuta la siguiente celda para imprimir las versiones de las bibliotecas clave utilizadas en este notebook. Copia esta salida a un archivo `requirements.txt` para asegurar la reproducibilidad del entorno.

In [ ]:
print("# requirements.txt\n")
print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
import sklearn
print(f"scikit-learn=={sklearn.__version__}")
print(f"pyarrow") # Necesario para to_parquet, la versión se puede fijar si es necesario